# Mixed-effects models of scene-level gaze metrics

This notebook tests whether depression severity predicts trial-level gaze
behaviour, and whether any such effect changes over the course of a session.
Each metric is fit as a linear mixed-effects model of the form

`metric ~ score * trial_norm + (1 + trial_norm | uid)`

The fixed-effects portion gives us three interpretable coefficients: the main effect of the depression score (does the metric differ between people with different severities?), the main effect of trial position within the session (does the metric drift across trials regardless of depression?), and their interaction (does the session-level trajectory differ between depression groups?). The random-effects structure lets each participant have their own baseline level and their own session-drift slope, so the fixed-effect estimates reflect the population-level pattern after accounting for stable individual differences.

### Two classes of metrics
- General metrics are computed across every clean stimulus scene regardless of which pair of images was shown. These are scanning-behaviour indicators - fixation and saccade rates, scanpath length, transition-matrix entropy, and related quantities - whose interpretation does not depend on scene content. Each of these gets one model fit on the full dataset.
- Pair-specific metrics attentional bias, per-valence dwell time, fixation proportion, revisit count - have different meaning depending on which two image categories were present. "Dwell on negative" is not the same attentional construct when the competitor is positive (threat vs reward) as when the competitor is neutral (threat vs baseline). Pooling across pair types would conflate these. Each per-pair metric therefore gets a separate fit per `scene_valence_pair`, and the resulting coefficients are labelled with both the metric name and the pair suffix.

Reported per model: score main effect, trial-position main effect, their
interaction (with FDR-corrected p-values), Cohen's d, ICC

In [0]:
%pip install statsmodels -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

from src.features.session_aggregation import PAIRS
from src.evaluation.lmm_temporal import fit_all_stratified, apply_fdr, plot_trajectories

## 1. Load data

In [0]:
PAIR_INVARIANT_METRICS = [
    "first_fixation_duration_ms",
    "mean_fixation_duration_ms",
    "fixation_rate_per_sec",
    "mean_saccade_amplitude",
    "saccade_rate_per_sec",
    "scanpath_length",
    "gaze_transition_entropy",
    "transition_matrix_density",
]

PAIR_STRATIFIED_METRICS = [
    "bias_neg_vs_pos",
    "bias_neg_vs_neu",
    "bias_pos_vs_neu",
    "dwell_time_ms_negative",
    "dwell_time_ms_positive",
    "dwell_time_ms_neutral",
    "dwell_time_500ms_negative",
    "dwell_time_500ms_positive",
    "dwell_time_500ms_neutral",
    "fixation_proportion_negative",
    "fixation_proportion_positive",
    "fixation_proportion_neutral",
    "revisit_count_negative",
    "revisit_count_positive",
    "revisit_count_neutral",
]

VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

cols = (["session_id", "scene_index", "trial_num", "scene_valence_pair"]
        + PAIR_INVARIANT_METRICS + PAIR_STRATIFIED_METRICS)

df = numbered.select(*cols).join(df_forms.select("session_id", "uid", "phq9_score", "bdi_score"), on="session_id", how="inner").toPandas()

df["trial_norm"] = df.groupby("session_id")["trial_num"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0
)
df["phq9_z"] = (df["phq9_score"] - df["phq9_score"].mean()) / df["phq9_score"].std()
df["bdi_z"] = (df["bdi_score"] - df["bdi_score"].mean()) / df["bdi_score"].std()

print(f"Rows: {len(df)}, users: {df['uid'].nunique()}")
print(f"General metrics: {len(PAIR_INVARIANT_METRICS)}")
print(f"Pair-specific metrics: {len(PAIR_STRATIFIED_METRICS)} × up to 3 pairs each")

## 2. PHQ-9

### 2.1 Fit all models

In [0]:
phq_results, phq_summary = fit_all_stratified(
    df, PAIR_INVARIANT_METRICS, PAIR_STRATIFIED_METRICS, "phq9_z", PAIRS
)
phq_summary = apply_fdr(phq_summary, "phq9_z")
print(f"Fits: {len(phq_results)}, summary rows: {len(phq_summary)}")
phq_summary[["metric", "pair_suffix", "phq9_z_coef", "phq9_z_pval_fdr", "phq9_z_d", "icc", "n_obs"]].sort_values("phq9_z_pval_fdr")

### 2.2 FDR-significant score effects (α = 0.05)

In [0]:
phq_summary[phq_summary["phq9_z_pval_fdr"] < 0.05][["metric", "pair_suffix", "phq9_z_coef", "phq9_z_pval_fdr", "phq9_z_d", "icc"]].sort_values("phq9_z_pval_fdr")

### 2.3 FDR-significant score × trial interactions

In [0]:
phq_summary[phq_summary["phq9_z:trial_norm_pval_fdr"] < 0.05][["metric", "pair_suffix", "phq9_z:trial_norm_coef", "phq9_z:trial_norm_pval_fdr"]]

### 2.4 Trajectories for general metrics

In [0]:
plot_trajectories(
    df, phq_summary, "phq9_z", "PHQ-9", "phq9_score", PAIR_INVARIANT_METRICS,
    save_individual=True,
    save_grid=True,
)

### 2.5 Trajectories for pair-specific metrics

In [0]:
for pair_name, pair_suffix, _ in PAIRS:
    pair_metrics = [m for m in PAIR_STRATIFIED_METRICS if phq_summary[(phq_summary["metric"] == m) & (phq_summary["pair_suffix"] == pair_suffix)].shape[0] > 0]
    plot_trajectories(
        df, phq_summary, "phq9_z", "PHQ-9", "phq9_score", pair_metrics,
        pair_filter=pair_name, pair_label=pair_suffix,
        save_individual=True,
        save_grid=True,
    )

## 3. BDI-II

### 3.1 Fit all models

In [0]:
bdi_results, bdi_summary = fit_all_stratified(df, PAIR_INVARIANT_METRICS, PAIR_STRATIFIED_METRICS, "bdi_z", PAIRS)
bdi_summary = apply_fdr(bdi_summary, "bdi_z")
bdi_summary[["metric", "pair_suffix", "bdi_z_coef", "bdi_z_pval_fdr", "bdi_z_d", "icc", "n_obs"]].sort_values("bdi_z_pval_fdr")

### 3.2 FDR-significant score effects (α = 0.05)

In [0]:
bdi_summary[bdi_summary["bdi_z_pval_fdr"] < 0.05][["metric", "pair_suffix", "bdi_z_coef", "bdi_z_pval_fdr", "bdi_z_d", "icc"]].sort_values("bdi_z_pval_fdr")

### 3.3 FDR-significant score × trial interactions

In [0]:
bdi_summary[bdi_summary["bdi_z:trial_norm_pval_fdr"] < 0.05][["metric", "pair_suffix", "bdi_z:trial_norm_coef", "bdi_z:trial_norm_pval_fdr"]]

### 3.4 Trajectories for general metrics

In [0]:
plot_trajectories(
    df, bdi_summary, "bdi_z", "BDI-II", "bdi_score", PAIR_INVARIANT_METRICS,
    save_individual=True,
    save_grid=True,
)

### 3.5 Trajectories for pair-specific metrics

In [0]:
for pair_name, pair_suffix, _ in PAIRS:
    pair_metrics = [m for m in PAIR_STRATIFIED_METRICS if bdi_summary[(bdi_summary["metric"] == m) & (bdi_summary["pair_suffix"] == pair_suffix)].shape[0] > 0]
    plot_trajectories(
        df, bdi_summary, "bdi_z", "BDI-II", "bdi_score", pair_metrics,
        pair_filter=pair_name, pair_label=pair_suffix,
        save_individual=True,
        save_grid=True,
    )

## 4. Detailed fits

In [0]:
KEY_KEYS = [
    "bias_neg_vs_pos__neg_vs_pos",
    "dwell_time_ms_negative__neg_vs_pos",
    "fixation_proportion_negative__neg_vs_pos",
    "dwell_time_ms_positive__pos_vs_neu",
]

for key in KEY_KEYS:
    if key not in phq_results:
        print(f"[skip {key}: not in results]")
        continue
    print()
    print(f"--- {key} (PHQ-9) ---")
    print()
    print(phq_results[key].summary())
    print()
    print(f"--- {key} (BDI-II) ---")
    print()
    print(bdi_results[key].summary())

## 5. PHQ-9 vs BDI-II comparison

In [0]:
comparison = phq_summary[["metric", "pair_suffix", "phq9_z_coef", "phq9_z_pval_fdr", "phq9_z_d", "icc"]].merge(bdi_summary[["metric", "pair_suffix", "bdi_z_coef", "bdi_z_pval_fdr", "bdi_z_d"]], on=["metric", "pair_suffix"]).sort_values("phq9_z_pval_fdr")
comparison

In [0]:
sig_phq = comparison["phq9_z_pval_fdr"] < 0.05
sig_bdi = comparison["bdi_z_pval_fdr"] < 0.05

pd.Series({
    "both": (sig_phq & sig_bdi).sum(),
    "phq_only": (sig_phq & ~sig_bdi).sum(),
    "bdi_only": (~sig_phq & sig_bdi).sum(),
    "neither": (~sig_phq & ~sig_bdi).sum(),
})

## 6. Reliability (ICC)

In [0]:
phq_summary[["metric", "pair_suffix", "icc"]].sort_values("icc", ascending=False)

In [0]:
phq_summary["icc"].agg(["mean", "median"]).round(3)